# 01 · Runtime, datos y evidencia diferencial

Este cuaderno es el único preflight. Lee exclusivamente desarrollo,
verifica el endpoint intrinsic4 y crea la evidencia clase-vs-rest dentro
de cada cohorte. No abre expresión ni etiquetas externas.

La evidencia no contiene firmas PAM50, MSigDB o DepMap. Su función es
proporcionar contexto transcriptómico reproducible a las aristas del
prior, no validar biología conocida.


In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "intergate").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Ejecute este notebook dentro de InterGATE_V8.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from intergate.config import CFG
from intergate.runtime_setup import (
    assert_v8_protocol, assert_transport_protocol, prepare_runtime,
)

assert_v8_protocol(CFG)
assert_transport_protocol(CFG)
ARTIFACTS = Path(CFG.ARTIFACTS_ROOT)
RUN_ROOT = ARTIFACTS / "INTERGATE_V8"
RESULTS = PROJECT_ROOT / "results_v8"
RESULTS.mkdir(parents=True, exist_ok=True)


In [ ]:
runtime = prepare_runtime(
    CFG,
    force_rebuild_backbone=False,
    force_rebuild_xh=False,
    force_rebuild_biology=False,
    build_biology_evidence=True,
    include_external=False,
)
assert runtime["external_opened"] is False
assert len(runtime["test_idx"]) == 0
assert tuple(runtime["classes"]) == tuple(CFG.MULTICLASS_LABELS_NO_CONTROL)

counts = (
    pd.DataFrame({"cohort": runtime["cohort"], "class": runtime["y_str"]})
    .value_counts().rename("n").reset_index()
    .pivot(index="cohort", columns="class", values="n").fillna(0).astype(int)
    .reindex(index=CFG.TRAIN_COHORTS, columns=runtime["classes"], fill_value=0)
)
display(counts)
print({
    "samples_development": int(len(runtime["y"])),
    "genes_connected": int(len(runtime["genes"])),
    "candidate_edges": int(runtime["candidate_edges"]),
    "kappa": float(runtime["kappa"]),
    "nominal_edges_per_class": int(runtime["k_nominal"]),
    "device": str(CFG.DEVICE),
})


In [ ]:
evidence = runtime["biology_evidence"]
if evidence is None:
    raise RuntimeError("No se construyó la evidencia biológica V8.")
group_counts = np.asarray(evidence["group_counts"])
audit_rows = []
for class_id, class_name in enumerate(runtime["classes"]):
    for cohort_id, cohort_name in enumerate(CFG.TRAIN_COHORTS):
        n_case, n_rest = group_counts[class_id, cohort_id]
        audit_rows.append({
            "class_name": class_name,
            "cohort": cohort_name,
            "n_case_train": int(n_case),
            "n_rest_train": int(n_rest),
            "eligible": bool(
                n_case >= CFG.BIOLOGY_MIN_SAMPLES_PER_GROUP
                and n_rest >= CFG.BIOLOGY_MIN_SAMPLES_PER_GROUP
            ),
        })
group_audit = pd.DataFrame(audit_rows)
display(group_audit)
if not group_audit.groupby("class_name").eligible.sum().ge(
    CFG.BIOLOGY_MIN_COHORTS
).all():
    raise RuntimeError("Alguna clase no alcanza el mínimo de cohortes.")

coverage = pd.DataFrame({
    "class_name": runtime["classes"],
    "cohorts_used": np.asarray(evidence["cohorts_used"], dtype=int),
    "nonzero_context_edges": [
        int(np.count_nonzero(evidence["context_score"][class_id]))
        for class_id in range(len(runtime["classes"]))
    ],
    "median_sign_consistency": [
        float(np.median(evidence["sign_consistency"][class_id]))
        for class_id in range(len(runtime["classes"]))
    ],
    "median_I2": [
        float(np.median(evidence["i2"][class_id]))
        for class_id in range(len(runtime["classes"]))
    ],
})
display(coverage)


In [ ]:
from intergate.biology import evidence_table

preflight = RESULTS / "01_preflight"
preflight.mkdir(parents=True, exist_ok=True)
group_audit.to_csv(preflight / "biology_group_counts.csv", index=False)
coverage.to_csv(preflight / "biology_evidence_coverage.csv", index=False)
evidence_table(
    evidence,
    edge_index=runtime["edge_index"],
    genes=runtime["genes"],
    edge_weight=runtime["edge_weight"],
    edge_type=runtime["edge_type"],
).to_csv(
    preflight / "biology_evidence_all_candidate_edges.csv.gz",
    index=False, compression="gzip",
)
summary = {
    "protocol_version": CFG.TRANSPORT_PROTOCOL_VERSION,
    "endpoint": list(runtime["classes"]),
    "development_cohorts": list(CFG.TRAIN_COHORTS),
    "external_opened": False,
    "n_samples": int(len(runtime["y"])),
    "n_genes": int(len(runtime["genes"])),
    "n_candidate_edges": int(runtime["candidate_edges"]),
    "kappa": float(runtime["kappa"]),
    "k_nominal": int(runtime["k_nominal"]),
    "known_biology_used_for_fitting": False,
}
(preflight / "preflight_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Preflight V8 completo:", preflight)
